In [1]:
import os
import json

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# obtain skeletons + label

DSET_PATH = r"./"
TRAIN_PATH = os.path.join(DSET_PATH, "train_skeletons")
TEST_PATH = os.path.join(DSET_PATH, "test_skeletons")

def load_skel_data(path):
    skeletons = torch.load(os.path.join(path, "skeletons_tensor.pt"))  
    with open(os.path.join(path, "skeleton_annots.json"), "r") as f:
        metadata = json.load(f)
    return skeletons, metadata

train_skels, train_metadata = load_skel_data(TRAIN_PATH)
# doing this rn cus ste didn't give test skels and metadata
test_skels, test_metadata = load_skel_data(TEST_PATH)

# minus one to make 0-indexed
train_labels = torch.tensor(
    [sample["label_id"] - 1 for sample in train_metadata["samples"]],
    dtype=torch.long,
)
test_labels = torch.tensor(
    [sample["label_id"] - 1 for sample in test_metadata["samples"]],
    dtype=torch.long,
)


# filter out first 100 training datapoints since they aren't labelled well
train_skels = train_skels[100:]
train_labels = train_labels[100:]

num_classes = len(torch.unique(train_labels))

# sanity check stuff
print("Labels shape:", train_labels.shape)
print("Skeletons shape:", train_skels.shape)
print("Num classes:", num_classes)

Labels shape: torch.Size([3781])
Skeletons shape: torch.Size([3781, 210, 21, 3])
Num classes: 14


In [3]:
train_ds = TensorDataset(train_skels, train_labels)
test_ds = TensorDataset(test_skels, test_labels)

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print("Train samples:", len(train_ds))
print("Test samples:", len(test_ds))

Train samples: 3781
Test samples: 1556


In [4]:
class MediapipeTransformer(nn.Module):
    def __init__(
        self,
        num_classes: int,
        num_joints: int = 21,
        coord_dim: int = 3,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 4,
        dim_feedforward: int = 256,
        max_seq_len: int = 128,
        dropout: float = 0.1,
    ):
        
        super().__init__()

        self.num_joints = num_joints
        self.coord_dim = coord_dim
        self.input_dim = num_joints * coord_dim
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # Project flattened skeleton frame -> d_model
        self.input_proj = nn.Linear(self.input_dim, d_model)

        # Positional embedding for time steps (including CLS)
        self.pos_embedding = nn.Embedding(max_seq_len + 1, d_model)

        # Learnable CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,   # (B, T, d_model)
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_classes)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        # (PyTorch already initializes Linear/Embedding reasonably)

    def forward(self, x):
        """
        x: (B, T, 21, 3)
        """
        B, T, J, C = x.shape
        assert J == self.num_joints and C == self.coord_dim, \
            f"Expected joints={self.num_joints}, coords={self.coord_dim}, got {J}, {C}"

        if T + 1 > self.max_seq_len + 1:
            raise ValueError(f"Sequence length {T} exceeds max_seq_len={self.max_seq_len}")

        # Flatten joints: (B, T, 21, 3) -> (B, T, 63)
        x = x.view(B, T, -1)

        # Project to d_model: (B, T, input_dim) -> (B, T, d_model)
        x = self.input_proj(x)

        # Create CLS tokens: (B, 1, d_model)
        cls_tokens = self.cls_token.expand(B, 1, self.d_model)

        # Prepend CLS: (B, T+1, d_model)
        x = torch.cat([cls_tokens, x], dim=1)

        # Positional embeddings for [0..T] (CLS at 0)
        positions = torch.arange(0, T + 1, device=x.device).unsqueeze(0)  # (1, T+1)
        pos_emb = self.pos_embedding(positions)                           # (1, T+1, d_model)
        x = x + pos_emb

        # Pass through Transformer encoder
        x = self.encoder(x)   # (B, T+1, d_model)

        # Use CLS token representation
        cls_repr = x[:, 0]    # (B, d_model)

        cls_repr = self.dropout(cls_repr)
        logits = self.fc(cls_repr)  # (B, num_classes)

        return logits

In [5]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == batch_y).sum().item()
        total_samples += batch_y.size(0)

    avg_loss = total_loss / total_samples
    acc = total_correct / total_samples
    return avg_loss, acc


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            total_loss += loss.item() * batch_x.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == batch_y).sum().item()
            total_samples += batch_y.size(0)

    avg_loss = total_loss / total_samples
    acc = total_correct / total_samples
    return avg_loss, acc


In [ ]:
device = "cpu"
model = MediapipeTransformer(
        num_classes=14,
        num_joints=21,
        d_model=128,
        nhead=4,
        num_layers=4,
        dim_feedforward=256,
        max_seq_len=240,     # must be >= T
        dropout=0.1,
    ).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

epochs = 15
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.3f} | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.3f}"
    )

ValueError: Sequence length 210 exceeds max_seq_len=64